# 하이브리드 모핑 파이프라인 — Colab 실행 노트북

로컬에서 만든 카메라 궤적(JSON)을 레포에서 받아, Blender로 depth/normal pass를 렌더하고,
ComfyUI로 실제 영상 프레임을 생성한 뒤, 디테일 복원 후 mp4로 출력한다.

**런타임: GPU (T4 이상) 필요** — 상단 메뉴 '런타임 > 런타임 유형 변경'에서 확인.

실행 순서: 0(클론) → 1(Blender) → 2(ComfyUI 설치) → 2d(Wan2.1-VACE-1.3B 모델, 기본 백엔드) → 2e(참조사진 흰배경 전처리, 로컬에서 안 했으면 여기서) → 2b/2c(SD1.5 레거시 폴백, 보통 생략) → 3(워크플로우 실행) → 4(디테일 복원) → 5(다운로드)


In [ ]:
# @title 0. 레포 클론 + 기본 설정
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/stereo-morph-pipeline.git"  # @param {type:"string"}
SHOT_ID = "demo_bullet_time"  # @param {type:"string"}

import os
%cd /content
!rm -rf stereo-morph-pipeline
!git clone $REPO_URL
%cd /content/stereo-morph-pipeline
!pip install -q -r requirements.txt

In [ ]:
# @title 1. Blender 설치 + 더미 메시 생성 + 카메라 패스 적용(depth/normal 렌더)
# Colab 기본 이미지엔 Blender가 없음 — 헤드리스 바이너리를 받아 설치
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz -O /tmp/blender.tar.xz
!tar -xf /tmp/blender.tar.xz -C /opt
!ln -sf /opt/blender-4.2.0-linux-x64/blender /usr/local/bin/blender
!blender --version

SHOT_JSON = f"/content/stereo-morph-pipeline/control/presets/{SHOT_ID}.json"
MESH_PATH = f"/content/dummy_subject_{SHOT_ID}.glb"
RENDER_DIR = f"/content/render/{SHOT_ID}"

!blender -b -P blender/generate_dummy_mesh.py -- --out $MESH_PATH
!blender -b -P blender/apply_camera_path.py -- --path $SHOT_JSON --mesh $MESH_PATH --out-dir $RENDER_DIR
print("depth/normal pass:", RENDER_DIR)

In [ ]:
# @title 2. ComfyUI + 커스텀 노드팩 설치
%cd /content
!rm -rf ComfyUI
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!rm -rf ComfyUI-AnimateDiff-Evolved ComfyUI-VideoHelperSuite
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git
!git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git  # 2026-10-23: 정체성 고정(FaceID) 실험
!pip install -q insightface onnxruntime  # IPAdapterFaceID/InsightFaceLoader 의존성

# 체크포인트/컨트롤넷/모션모듈 가중치는 용량이 커서 Drive 마운트 후 심링크 권장.
# 이전 세션의 죽은 마운트 찌꺼기가 남아있으면 force_remount도 실패할 수 있어
# 그 경우 로컬 마운트포인트를 지우고 재시도한다 (클라우드 쪽 Drive 데이터는 안전함).
import shutil
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except ValueError:
    print('마운트 찌꺼기 정리 후 재시도')
    shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive', force_remount=True)

In [ ]:
# @title 2d. Wan2.1-VACE-1.3B 모델 다운로드 (기본 백엔드) — 2b/2c(SD1.5 레거시)는 보통 생략 가능
# 2026-10-24: 네이티브 ComfyUI 코어 노드(WanVaceToVideo)만 쓰므로 커스텀 노드팩 설치 불필요.
# 파일 경로/URL은 ComfyUI 공식 템플릿(video_wan_vace_14B_v2v.json, comfy-core 0.3.34)과
# comfy_extras/nodes_wan.py 소스를 직접 받아 대조 확인함 — 추측 아님.
import os

DRIVE_CACHE = "/content/drive/MyDrive/morph-pipeline-models"
os.makedirs(DRIVE_CACHE, exist_ok=True)

VACE_MODELS = [
    ("diffusion_models", "wan2.1_vace_1.3B_fp16.safetensors",
     "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_vace_1.3B_fp16.safetensors"),
    ("vae", "wan_2.1_vae.safetensors",
     "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors"),
    ("text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
]

for subdir, filename, url in VACE_MODELS:
    target_dir = f"/content/ComfyUI/models/{subdir}"
    os.makedirs(target_dir, exist_ok=True)
    target_path = f"{target_dir}/{filename}"
    cache_path = f"{DRIVE_CACHE}/{filename}"

    if os.path.exists(target_path):
        print(f"이미 있음: {target_path}")
    elif os.path.exists(cache_path):
        print(f"Drive 캐시에서 복사: {filename}")
        !cp "{cache_path}" "{target_path}"
    else:
        print(f"다운로드: {filename} (최초 1회만 느림, ~11GB 총합)")
        !wget -q --show-progress -O "{target_path}" "{url}"
        print(f"Drive에 캐시 저장: {filename}")
        !cp "{target_path}" "{cache_path}"

print()
print("Wan2.1-VACE-1.3B 모델 준비 완료 (diffusion_models + vae + text_encoders)")


In [ ]:
# @title 2e. 참조사진 배경 제거 (rembg) — 로컬에서 input/preprocess_reference.py로 이미
# CP2(output/<shot_id>/cp2_hero_frame/hero.png)를 만들어 그 파일을 바로 업로드했다면 이 셀은 건너뛸 것.
!pip install -q rembg

import os
from PIL import Image
from rembg import remove

RAW_SUBJECT_IMAGE = "/content/subject_ref_raw.png"  # 배경 안 지운 원본 업로드 파일명
CLEAN_SUBJECT_IMAGE = "/content/subject_ref_clean.png"

if os.path.exists(CLEAN_SUBJECT_IMAGE):
    print(f"이미 있음(로컬에서 전처리된 파일을 업로드한 경우): {CLEAN_SUBJECT_IMAGE}")
elif os.path.exists(RAW_SUBJECT_IMAGE):
    src = Image.open(RAW_SUBJECT_IMAGE).convert("RGBA")
    cut = remove(src)
    white_bg = Image.new("RGBA", cut.size, (255, 255, 255, 255))
    white_bg.paste(cut, mask=cut.split()[3])
    white_bg.convert("RGB").save(CLEAN_SUBJECT_IMAGE)
    print(f"흰 배경 참조 이미지 저장: {CLEAN_SUBJECT_IMAGE}")
else:
    raise FileNotFoundError(
        f"{RAW_SUBJECT_IMAGE} 도 {CLEAN_SUBJECT_IMAGE} 도 없음 — "
        "Colab 파일 패널에 둘 중 하나를 먼저 업로드할 것"
    )


In [ ]:
# @title 2b. 모델 가중치 받기 (체크포인트/ControlNet/AnimateDiff 모션모듈) - 최초 1회만 느림, 이후 Drive 캐시로 빠름
import os

DRIVE_CACHE = "/content/drive/MyDrive/morph-pipeline-models"
os.makedirs(DRIVE_CACHE, exist_ok=True)

MODELS = [
    ("checkpoints", "v1-5-pruned-emaonly.safetensors",
     "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"),
    ("controlnet", "control_v11f1p_sd15_depth.pth",
     "https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth"),
    ("animatediff_models", "mm_sd_v15_v2.ckpt",
     "https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt"),
]

for subdir, filename, url in MODELS:
    target_dir = f"/content/ComfyUI/models/{subdir}"
    os.makedirs(target_dir, exist_ok=True)
    target_path = f"{target_dir}/{filename}"
    cache_path = f"{DRIVE_CACHE}/{filename}"

    if os.path.exists(target_path):
        print(f"이미 있음: {target_path}")
    elif os.path.exists(cache_path):
        print(f"Drive 캐시에서 복사: {filename}")
        !cp "{cache_path}" "{target_path}"
    else:
        print(f"다운로드: {filename}")
        !wget -q --show-progress -O "{target_path}" "{url}"
        print(f"Drive에 캐시 저장: {filename}")
        !cp "{target_path}" "{cache_path}"

# AnimateDiff-Evolved는 모션모듈을 custom_nodes 쪽 models/ 폴더도 봅니다 - 심링크로 양쪽 커버
ade_models_dir = "/content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models"
os.makedirs(ade_models_dir, exist_ok=True)
!ln -sf /content/ComfyUI/models/animatediff_models/mm_sd_v15_v2.ckpt "{ade_models_dir}/mm_sd_v15_v2.ckpt"

print()
print("모델 준비 완료 (체크포인트/ControlNet/모션모듈 전부 SD1.5 계열로 통일)")

In [ ]:
# @title 2c. IP-Adapter FaceID 모델 받기 (2026-10-23 "정체성 유지" 실험용 — 선택)
# 아래 3번 셀에서 USE_IP_ADAPTER_FACEID=True 로 켤 때만 필요. 2b와 같은 Drive 캐시 방식.
import os

DRIVE_CACHE = "/content/drive/MyDrive/morph-pipeline-models"

IP_ADAPTER_MODELS = [
    ("ipadapter", "ip-adapter-faceid_sd15.bin",
     "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid_sd15.bin"),
    ("loras", "ip-adapter-faceid_sd15_lora.safetensors",
     "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid_sd15_lora.safetensors"),
    # 2026-10-28: IPAdapterFaceID가 "Missing CLIPVision model." 로 실패해서 추가(Colab 실측) -
    # ip-adapter-faceid_sd15.bin 자체엔 clip vision 가중치가 없어서 별도 로더가 필요함.
    ("clip_vision", "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors",
     "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors"),
]

for subdir, filename, url in IP_ADAPTER_MODELS:
    target_dir = f"/content/ComfyUI/models/{subdir}"
    os.makedirs(target_dir, exist_ok=True)
    target_path = f"{target_dir}/{filename}"
    cache_path = f"{DRIVE_CACHE}/{filename}"

    if os.path.exists(target_path):
        print(f"이미 있음: {target_path}")
    elif os.path.exists(cache_path):
        print(f"Drive 캐시에서 복사: {filename}")
        !cp "{cache_path}" "{target_path}"
    else:
        print(f"다운로드: {filename}")
        !wget -q --show-progress -O "{target_path}" "{url}"
        print(f"Drive에 캐시 저장: {filename}")
        !cp "{target_path}" "{cache_path}"

# insightface 자체 얼굴분석 모델(buffalo_l, ~300MB)은 InsightFaceLoader 노드가
# 처음 쓰일 때 자동 다운로드함(~/.insightface 캐시) — 여기선 안 받아도 됨.
print()
print("IP-Adapter FaceID 모델 준비 완료")

In [ ]:
# @title (검증) 설치된 노드 class_type 확인 — workflow_builder.py 와 이름이 다르면 여기서 맞춰야 함
import subprocess, threading, time, requests

def start_comfyui():
    subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"], cwd="/content/ComfyUI")

threading.Thread(target=start_comfyui, daemon=True).start()

# 고정 대기(이전엔 time.sleep(15))는 커스텀 노드가 늘어날수록 import 시간이 길어져
# ConnectionRefusedError로 반복 실패했음(2026-10-22~23 실측) — 재시도 폴링으로 교체.
ok = False
for i in range(20):  # 최대 60초, 3초씩
    time.sleep(3)
    try:
        resp = requests.get("http://127.0.0.1:8188/object_info", timeout=2)
        if resp.status_code == 200:
            ok = True
            break
    except requests.exceptions.ConnectionError:
        print(f"대기 중... ({(i+1)*3}초)")

if ok:
    needed = [
        # wan_vace 백엔드(기본, 2026-10-24~) — 네이티브 ComfyUI 코어 노드, 커스텀 노드팩 불필요
        "UNETLoader", "CLIPLoader", "VAELoader", "ModelSamplingSD3", "WanVaceToVideo", "TrimVideoLatent",
        "VHS_LoadImagesPath", "VHS_VideoCombine",
        # sd15_animatediff 레거시 백엔드(폴백) — 아래는 그 경로를 쓸 때만 필요
        "ADE_AnimateDiffLoaderGen1", "ACN_ControlNetLoaderAdvanced", "ACN_AdvancedControlNetApply",
        "IPAdapterModelLoader", "IPAdapterInsightFaceLoader", "IPAdapterFaceID", "LoraLoaderModelOnly",
    ]
    resp_json = resp.json()
    for n in needed:
        print(n, "OK" if n in resp_json else "⚠️ 이름이 다름 — comfyui/workflow_builder.py 수정 필요")
else:
    print("60초 넘게 응답 없음 — 서버가 죽었을 수 있음, 로그 확인 필요")


In [ ]:
# @title 3. 워크플로우 조립 + 큐 전송
import sys, json, requests, shutil, os
sys.path.insert(0, "/content/stereo-morph-pipeline/comfyui")
from workflow_builder import write_workflow_json

BACKEND = "wan_vace"  # @param ["wan_vace", "sd15_animatediff"]
REFERENCE_STRENGTH = 1.2  # @param {type:"number"}  -- WanVaceToVideo strength, 1.0~1.5 권장
USE_IP_ADAPTER_FACEID = True  # @param {type:"boolean"}  -- BACKEND=="sd15_animatediff"일 때만 의미 있음

if BACKEND == "wan_vace":
    SUBJECT_IMAGE_SRC = "/content/subject_ref_clean.png"  # 2d/2e에서 만든 흰 배경 참조 이미지
else:
    SUBJECT_IMAGE_SRC = "/content/subject_ref.png"  # SD1.5 레거시 경로는 전처리 없이 원본 그대로

if not os.path.exists(SUBJECT_IMAGE_SRC):
    raise FileNotFoundError(
        f"{SUBJECT_IMAGE_SRC} 없음 — 2d/2e 셀을 먼저 실행했는지, 또는 Colab 파일 패널에 "
        "업로드했는지 확인할 것"
    )

# ComfyUI LoadImage는 절대경로가 아니라 자기 input/ 폴더 안의 파일명만 받음(2026-10-23 실측).
SUBJECT_IMAGE = os.path.basename(SUBJECT_IMAGE_SRC)
shutil.copy(SUBJECT_IMAGE_SRC, f"/content/ComfyUI/input/{SUBJECT_IMAGE}")

workflow_path = f"/content/stereo-morph-pipeline/comfyui/workflows/{SHOT_ID}.json"
if os.path.exists(workflow_path):
    # 로컬 build_shot.py가 이미 만들어서 git push한 워크플로우가 있으면 그걸 그대로 사용
    # (reference_strength 등 로컬에서 정한 파라미터가 그대로 반영됨).
    with open(workflow_path, encoding="utf-8") as f:
        wf = json.load(f)
    print(f"기존 워크플로우 로드: {workflow_path}")
else:
    write_workflow_json(
        workflow_path,
        backend=BACKEND,
        shot_id=SHOT_ID,
        depth_pass_dir=f"{RENDER_DIR}/depth",
        subject_image_path=SUBJECT_IMAGE,
        positive_prompt="cinematic photoreal subject, bullet time orbit shot, consistent identity, sharp focus",
        **({"reference_strength": REFERENCE_STRENGTH} if BACKEND == "wan_vace" else {"use_ip_adapter_faceid": USE_IP_ADAPTER_FACEID}),
    )
    with open(workflow_path, encoding="utf-8") as f:
        wf = json.load(f)

r = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": wf})
print(r.status_code, r.text[:500])


In [ ]:
# @title 4. 디테일 복원 (옵션 — 필요한 단계만 레포 경로 채우고 실행)
sys.path.insert(0, "/content/stereo-morph-pipeline/refinement")
from refine import refine_chain

RAW_FRAMES_DIR = "/content/ComfyUI/output"  # ComfyUI가 뱉은 생성 프레임 폴더로 수정

refine_chain(
    raw_frames_dir=RAW_FRAMES_DIR,
    work_dir=f"/content/refine_work/{SHOT_ID}",
    final_out_path=f"/content/drive/MyDrive/morph-pipeline-output/{SHOT_ID}.mp4",
    fps=24,
    rife_repo_dir=None,        # 필요 시 ECCV2022-RIFE 클론 경로
    codeformer_repo_dir=None,  # 필요 시 CodeFormer 클론 경로
    realesrgan_repo_dir=None,  # 필요 시 Real-ESRGAN 클론 경로
)

In [ ]:
# @title 5. 결과 다운로드 (로컬 모니터 앱용)
# Drive 동기화 앱이 없어도 되는 방식 — 브라우저로 바로 .mp4 다운로드 트리거.
# 받은 파일을 PC의 output/<SHOT_ID>/cp3_segments/segment_01.mp4 로 넣으면
# server/app.py 모니터(localhost:4100)에서 바로 보임.
# (CP2 히어로 프레임은 IP-Adapter FaceID 실험에서 별도로 추가될 예정 — 아직 없음)
from google.colab import files

final_mp4 = f"/content/drive/MyDrive/morph-pipeline-output/{SHOT_ID}.mp4"
print(f"다운로드 대상: {final_mp4}")
files.download(final_mp4)
print()
print(f"받은 파일을 output/{SHOT_ID}/cp3_segments/segment_01.mp4 로 저장 후 localhost:4100 새로고침")

## 5. 출력
결과 mp4는 `Google Drive > morph-pipeline-output/`에 저장되고, 위 셀에서 브라우저로도
바로 다운로드된다. 받은 파일은 로컬 PC의 `output/<SHOT_ID>/cp3_segments/segment_01.mp4`
로 넣으면 `server/app.py` 모니터(`localhost:4100`)에서 바로 확인 가능.

실행 순서는 이제 0 → 1 → 2 → 2b → 3 → 4 → **5(다운로드)**.